<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-16/notebooks/ClimatePipeline/02_Climate_TemperaturaMinima_DataAudit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Climate_TemperaturaMinima_DataAudit

Auditoría genérica y de solo lectura para los Parquet climáticos del proyecto RAIZ.

Este notebook no limpia ni transforma los datos crudos. Produce un diagnóstico reproducible para definir las reglas de limpieza y preparar la construcción posterior de registros diarios.

## Alcance del diagnóstico

- Inventario exacto mediante metadatos Parquet.
- Validación de esquema y tipos entre archivos.
- Muestras estratificadas por departamento, año y mes.
- Bloques contiguos de archivos para estudiar frecuencia sin mezclar saltos artificiales.
- Nulos, conversiones fallidas, unidades, valores y coordenadas.
- Duplicados exactos, duplicados de clave y conflictos.
- Cobertura y frecuencia por estación y sensor.
- Conteo completo opcional por estación y sensor.
- Hallazgos clasificados por severidad.

La auditoría no imputa datos ni decide una agregación diaria. Esas reglas dependen de la variable y pertenecen al siguiente componente del pipeline.


## 1. Configuración

Configure una variable y una sola combinación manejable de departamentos, años y meses. Para las auditorías históricas conviene ejecutar un departamento y un año por corrida.

Todas las banderas de ejecución permanecen en `False` para que **Run all sea seguro**. Las celdas definen funciones, pero no recorren Drive hasta activar exactamente un modo.

### Recetas de ejecución

- Auditoría histórica por muestra: `EJECUTAR_AUDITORIA=True`, `EJECUTAR_CONTEO_ESTACIONES_COMPLETO=False` y las dos banderas de inventario en `False`.
- Auditoría con conteo completo: igual que la anterior, pero con `EJECUTAR_CONTEO_ESTACIONES_COMPLETO=True`.
- Inventario general aproximado: solo `EJECUTAR_INVENTARIO_GENERAL=True`.
- Inventario operativo final: solo `EJECUTAR_INVENTARIO_OPERATIVO=True`.

`GUARDAR_RESULTADOS=True` únicamente escribe cuando la auditoría está activa. El conteo completo es opcional y no se necesita para las muestras históricas preliminares de 2021 y 2023.


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-16'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        remote_ref = f'refs/remotes/origin/{REPO_REF}'
        subprocess.run(
            ['git', 'fetch', '--depth', '1', 'origin', f'+refs/heads/{REPO_REF}:{remote_ref}'],
            cwd=REPO_DIR,
            check=True,
        )
        subprocess.run(
            ['git', 'checkout', '-B', REPO_REF, remote_ref],
            cwd=REPO_DIR,
            check=True,
        )
    PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
else:
    candidatos = [
        REPO_DIR / 'notebooks' / 'ClimatePipeline',
        REPO_DIR / 'ClimatePipeline',
        REPO_DIR,
        REPO_DIR.parent / 'ClimatePipeline',
    ]
    PIPELINE_DIR = next(
        (ruta for ruta in candidatos if (ruta / 'DatasetConfig.py').exists()),
        None,
    )
if PIPELINE_DIR is None or not PIPELINE_DIR.exists():
    raise FileNotFoundError('No se encontró notebooks/ClimatePipeline/DatasetConfig.py.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from DatasetConfig import cargar_configuracion_datasets

DATASET_CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)
PROCESSED_ROOT = DATASET_CONFIG.processed_root

# Deben coincidir con la carpeta creada por 01_Climate_TemperaturaMinima_DataDownloader.
DATASET_ID = 'afdg-3zpb'
VARIABLE_NOMBRE = 'temperatura_minima'

# Para las muestras históricas, cambie un departamento y un año por corrida.
AUDITORIA_DEPARTAMENTOS = ['CUNDINAMARCA', 'BOYACÁ']
AUDITORIA_ANIOS = [2024, 2025]
AUDITORIA_MESES = None  # None usa todos los meses disponibles.

# Muestra: dos bloques aleatorios de dos archivos contiguos por partición.
BLOQUES_POR_PARTICION = 2
ARCHIVOS_POR_BLOQUE = 2
MAX_FILAS_MUESTRA = 250_000
SEMILLA_MUESTRA = 2026

# None solo describe la distribución. Ejemplo para otra variable: (300, 1100).
RANGO_PLAUSIBLE = None

# Cadencias observadas de precipitación: 1, 2, 5, 10 y 60 minutos.
# Use None si aún no se ha definido el contrato de otra variable.
CADENCIAS_ESPERADAS_SEGUNDOS = [60, 120, 300, 600, 3600]
UMBRAL_VARIACION_COORDENADAS_METROS = 100

# Modos protegidos. Active exactamente uno para ejecutar Run all.
EJECUTAR_AUDITORIA = False
EJECUTAR_CONTEO_ESTACIONES_COMPLETO = False
EJECUTAR_INVENTARIO_GENERAL = False
EJECUTAR_INVENTARIO_OPERATIVO = False

# Solo tiene efecto cuando EJECUTAR_AUDITORIA=True.
GUARDAR_RESULTADOS = True
ETIQUETA_SALIDA = None  # None genera una etiqueta informativa automáticamente.

modos_independientes = {
    'auditoria': EJECUTAR_AUDITORIA,
    'inventario_general': EJECUTAR_INVENTARIO_GENERAL,
    'inventario_operativo': EJECUTAR_INVENTARIO_OPERATIVO,
}
activos = [nombre for nombre, activo in modos_independientes.items() if activo]
if len(activos) > 1:
    raise ValueError(f'Active un solo modo de ejecución; activos: {activos}.')
if EJECUTAR_CONTEO_ESTACIONES_COMPLETO and not EJECUTAR_AUDITORIA:
    raise ValueError(
        'El conteo completo requiere EJECUTAR_AUDITORIA=True.'
    )

if EJECUTAR_AUDITORIA and EJECUTAR_CONTEO_ESTACIONES_COMPLETO:
    MODO_EJECUCION = 'auditoria_con_conteo_completo'
elif EJECUTAR_AUDITORIA:
    MODO_EJECUCION = 'auditoria_muestra'
elif EJECUTAR_INVENTARIO_GENERAL:
    MODO_EJECUCION = 'inventario_general'
elif EJECUTAR_INVENTARIO_OPERATIVO:
    MODO_EJECUCION = 'inventario_operativo'
else:
    MODO_EJECUCION = 'desactivado_seguro'

COLUMNAS_ESPERADAS = [
    'codigoestacion',
    'codigosensor',
    'dataset_id',
    'departamento',
    'descripcionsensor',
    'fechaobservacion',
    'latitud',
    'longitud',
    'municipio',
    'nombreestacion',
    'unidadmedida',
    'valorobservado',
    'zonahidrografica',
]

print({
    'modo': MODO_EJECUCION,
    'dataset_id': DATASET_ID,
    'variable': VARIABLE_NOMBRE,
    'departamentos': AUDITORIA_DEPARTAMENTOS,
    'anios': AUDITORIA_ANIOS,
    'meses': AUDITORIA_MESES,
    'bloques_por_particion': BLOQUES_POR_PARTICION,
    'archivos_por_bloque': ARCHIVOS_POR_BLOQUE,
    'conteo_estaciones_completo': EJECUTAR_CONTEO_ESTACIONES_COMPLETO,
    'inventario_general': EJECUTAR_INVENTARIO_GENERAL,
    'inventario_operativo': EJECUTAR_INVENTARIO_OPERATIVO,
    'guardar_resultados': GUARDAR_RESULTADOS,
    'ejecutar_auditoria': EJECUTAR_AUDITORIA,
    'processed_root': str(PROCESSED_ROOT),
})


## 2. Inventario general aproximado

Este inventario recorre todas las variables dentro de `clima_crudo`, pero no abre los Parquet. Cuenta archivos `part-*` y usa el máximo de 1.000 filas por archivo para producir una estimación superior rápida.

Los años y departamentos se deducen de la estructura de carpetas. El total real puede ser menor porque el último archivo de cada partición suele contener menos de 1.000 filas.


In [ ]:
import unicodedata


def normalizar_unicode(valor):
    return unicodedata.normalize('NFC', str(valor))


def etiquetas_desde_ruta(archivo, raiz):
    etiquetas = {}
    for parte in archivo.relative_to(raiz).parts:
        if '=' in parte:
            clave, valor = parte.split('=', 1)
            etiquetas[clave] = (
                normalizar_unicode(valor) if clave == 'departamento' else valor
            )
    return etiquetas


def construir_inventario_general(raiz_clima, filas_maximas_por_archivo=1000):
    if not raiz_clima.exists():
        return pd.DataFrame()

    acumulado = {}
    for archivo in raiz_clima.rglob('part-*.parquet'):
        etiquetas = etiquetas_desde_ruta(archivo, raiz_clima)
        variable = etiquetas.get('variable')
        fuente = etiquetas.get('fuente')
        if not variable or not fuente:
            continue

        clave = (variable, fuente)
        if clave not in acumulado:
            acumulado[clave] = {
                'lotes_archivos': 0,
                'anios': set(),
                'departamentos': set(),
            }

        actual = acumulado[clave]
        actual['lotes_archivos'] += 1
        if str(etiquetas.get('anio', '')).isdigit():
            actual['anios'].add(int(etiquetas['anio']))
        if etiquetas.get('departamento'):
            actual['departamentos'].add(etiquetas['departamento'])

    filas = []
    for (variable, fuente), valores in sorted(acumulado.items()):
        anios = sorted(valores['anios'])
        estimado = valores['lotes_archivos'] * int(filas_maximas_por_archivo)
        filas.append({
            'variable': variable,
            'fuente': fuente,
            'lotes_archivos': valores['lotes_archivos'],
            'registros_estimados': f'~{estimado:,}',
            'registros_estimados_max': estimado,
            'anio_inicio': anios[0] if anios else None,
            'anio_fin': anios[-1] if anios else None,
            'departamentos': ', '.join(sorted(valores['departamentos'])),
        })

    return pd.DataFrame(filas)


inventario_general_descargas = pd.DataFrame()
if not EJECUTAR_INVENTARIO_GENERAL:
    print(
        'Inventario general desactivado. Cambie '
        'EJECUTAR_INVENTARIO_GENERAL a True para generar la tabla.'
    )
else:
    raiz_clima = PROCESSED_ROOT / 'clima_crudo'
    inventario_general_descargas = construir_inventario_general(raiz_clima)
    if inventario_general_descargas.empty:
        print(f'No se encontraron archivos part-*.parquet en {raiz_clima}.')
    else:
        display(inventario_general_descargas)


## 3. Descubrimiento e inventario Parquet

El inventario recorre los archivos seleccionados y consulta sus metadatos sin cargar las filas. Los conteos de archivos, filas y bytes de esta sección son exactos.

También se comprueba la secuencia de archivos part-xxxxx.parquet dentro de cada partición.


In [ ]:
import json
import random
import re
import unicodedata
from collections import defaultdict

PART_PATTERN = re.compile(r'^part-(\d{5})\.parquet$')
DEPARTAMENTOS_PERMITIDOS = {'BOYACÁ', 'CUNDINAMARCA'}


def slugificar(valor):
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('ascii').lower()
    texto = re.sub(r'[^a-z0-9]+', '_', texto).strip('_')
    if not texto:
        raise ValueError(f'No se pudo construir una etiqueta para {valor!r}.')
    return texto


def normalizar_lista_enteros(valores, minimo, maximo, nombre):
    if valores is None:
        return None
    normalizados = sorted({int(valor) for valor in valores})
    fuera_rango = [valor for valor in normalizados if valor < minimo or valor > maximo]
    if fuera_rango:
        raise ValueError(f'{nombre} fuera de rango: {fuera_rango}.')
    return normalizados


def validar_configuracion():
    departamentos = sorted({
        normalizar_unicode(str(departamento).strip().upper())
        for departamento in AUDITORIA_DEPARTAMENTOS
    })
    if not departamentos:
        raise ValueError('AUDITORIA_DEPARTAMENTOS no puede estar vacío.')

    no_permitidos = set(departamentos) - DEPARTAMENTOS_PERMITIDOS
    if no_permitidos:
        raise ValueError(f'Departamentos fuera del alcance: {sorted(no_permitidos)}.')

    anios = normalizar_lista_enteros(AUDITORIA_ANIOS, 1900, 2100, 'Años')
    meses = normalizar_lista_enteros(AUDITORIA_MESES, 1, 12, 'Meses')

    if int(BLOQUES_POR_PARTICION) <= 0:
        raise ValueError('BLOQUES_POR_PARTICION debe ser positivo.')
    if int(ARCHIVOS_POR_BLOQUE) <= 0:
        raise ValueError('ARCHIVOS_POR_BLOQUE debe ser positivo.')
    if int(MAX_FILAS_MUESTRA) <= 0:
        raise ValueError('MAX_FILAS_MUESTRA debe ser positivo.')
    if RANGO_PLAUSIBLE is not None:
        if len(RANGO_PLAUSIBLE) != 2 or RANGO_PLAUSIBLE[0] >= RANGO_PLAUSIBLE[1]:
            raise ValueError('RANGO_PLAUSIBLE debe ser None o una pareja (mínimo, máximo).')

    return departamentos, anios, meses


def raiz_dataset():
    return (
        PROCESSED_ROOT
        / 'clima_crudo'
        / f'variable={slugificar(VARIABLE_NOMBRE)}'
        / f'fuente={str(DATASET_ID).lower()}'
    )


def particiones_desde_ruta(archivo, raiz):
    valores = {}
    for parte in archivo.relative_to(raiz).parts:
        if '=' in parte:
            clave, valor = parte.split('=', 1)
            valores[clave] = valor

    try:
        anio = int(valores.get('anio'))
        mes = int(valores.get('mes'))
    except (TypeError, ValueError):
        anio = None
        mes = None

    return {
        'departamento': (
            normalizar_unicode(valores.get('departamento'))
            if valores.get('departamento') is not None
            else None
        ),
        'anio': anio,
        'mes': mes,
    }


def descubrir_archivos(raiz, departamentos, anios=None, meses=None):
    if not raiz.exists():
        return []

    archivos = []
    for archivo in raiz.rglob('*.parquet'):
        particion = particiones_desde_ruta(archivo, raiz)
        if particion['departamento'] not in departamentos:
            continue
        if anios is not None and particion['anio'] not in anios:
            continue
        if meses is not None and particion['mes'] not in meses:
            continue
        archivos.append(archivo)

    return sorted(
        archivos,
        key=lambda archivo: (
            particiones_desde_ruta(archivo, raiz)['departamento'] or '',
            particiones_desde_ruta(archivo, raiz)['anio'] or -1,
            particiones_desde_ruta(archivo, raiz)['mes'] or -1,
            archivo.name,
        ),
    )


def construir_inventario(archivos, raiz):
    try:
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise ImportError('Se necesita pyarrow para auditar Parquet.') from exc

    filas = []
    for archivo in archivos:
        particion = particiones_desde_ruta(archivo, raiz)
        coincidencia = PART_PATTERN.fullmatch(archivo.name)

        try:
            parquet = pq.ParquetFile(archivo)
            esquema = {
                campo.name: str(campo.type)
                for campo in parquet.schema_arrow
            }
            filas.append({
                **particion,
                'parte': int(coincidencia.group(1)) if coincidencia else None,
                'archivo': str(archivo),
                'filas': parquet.metadata.num_rows,
                'tamano_bytes': archivo.stat().st_size,
                'numero_columnas': len(esquema),
                'columnas': tuple(esquema),
                'esquema': esquema,
                'firma_esquema': json.dumps(esquema, sort_keys=True, ensure_ascii=False),
                'error_lectura': None,
            })
        except Exception as exc:
            filas.append({
                **particion,
                'parte': int(coincidencia.group(1)) if coincidencia else None,
                'archivo': str(archivo),
                'filas': None,
                'tamano_bytes': archivo.stat().st_size if archivo.exists() else None,
                'numero_columnas': None,
                'columnas': tuple(),
                'esquema': {},
                'firma_esquema': None,
                'error_lectura': f'{type(exc).__name__}: {exc}',
            })

    return pd.DataFrame(filas)


def resumir_particiones(inventario):
    if inventario.empty:
        return pd.DataFrame()

    registros = []
    claves = ['departamento', 'anio', 'mes']
    for clave, grupo in inventario.groupby(claves, dropna=False, sort=True):
        partes = sorted(grupo['parte'].dropna().astype(int).tolist())
        faltantes = []
        if partes:
            esperadas = set(range(partes[-1] + 1))
            faltantes = sorted(esperadas - set(partes))

        registros.append({
            **dict(zip(claves, clave)),
            'archivos': len(grupo),
            'filas': int(grupo['filas'].fillna(0).sum()),
            'tamano_mb': round(grupo['tamano_bytes'].fillna(0).sum() / (1024 ** 2), 2),
            'primera_parte': partes[0] if partes else None,
            'ultima_parte': partes[-1] if partes else None,
            'partes_faltantes': faltantes,
            'errores_lectura': int(grupo['error_lectura'].notna().sum()),
        })

    return pd.DataFrame(registros)


def resumir_esquemas(inventario):
    if inventario.empty:
        return pd.DataFrame(), pd.DataFrame()

    archivos_validos = inventario[inventario['error_lectura'].isna()]
    presencia = []
    tipos = []

    for columna in sorted(set().union(*archivos_validos['columnas'].tolist())):
        presencia.append({
            'columna': columna,
            'archivos_presente': int(archivos_validos['columnas'].apply(
                lambda columnas: columna in columnas
            ).sum()),
            'archivos_totales': len(archivos_validos),
            'esperada': columna in COLUMNAS_ESPERADAS,
        })

        conteo_tipos = defaultdict(int)
        for esquema in archivos_validos['esquema']:
            if columna in esquema:
                conteo_tipos[esquema[columna]] += 1
        for tipo, cantidad in sorted(conteo_tipos.items()):
            tipos.append({
                'columna': columna,
                'tipo_parquet': tipo,
                'archivos': cantidad,
            })

    return pd.DataFrame(presencia), pd.DataFrame(tipos)


## 4. Selección de muestras estratificadas

Cada combinación departamento + año + mes es un estrato. Dentro de cada estrato se seleccionan bloques aleatorios de archivos contiguos.

La continuidad dentro de cada bloque permite estimar intervalos temporales sin interpretar como frecuencia el salto existente entre dos muestras alejadas. La semilla hace reproducible la selección.


In [ ]:
def seleccionar_archivos_muestra(inventario):
    validos = inventario[
        inventario['error_lectura'].isna()
        & inventario['parte'].notna()
    ].copy()
    if validos.empty:
        return pd.DataFrame()

    rng = random.Random(int(SEMILLA_MUESTRA))
    seleccion = []
    claves = ['departamento', 'anio', 'mes']

    for clave, grupo in validos.groupby(claves, sort=True):
        grupo = grupo.sort_values('parte').reset_index(drop=True)
        cantidad = len(grupo)
        tamano_bloque = min(int(ARCHIVOS_POR_BLOQUE), cantidad)
        candidatos = list(range(0, cantidad - tamano_bloque + 1))
        rng.shuffle(candidatos)

        ocupados = set()
        inicios = []
        for inicio in candidatos:
            indices = set(range(inicio, inicio + tamano_bloque))
            if indices & ocupados:
                continue
            inicios.append(inicio)
            ocupados.update(indices)
            if len(inicios) >= int(BLOQUES_POR_PARTICION):
                break

        if not inicios:
            inicios = [0]

        for numero_bloque, inicio in enumerate(sorted(inicios), start=1):
            bloque = grupo.iloc[inicio:inicio + tamano_bloque].copy()
            bloque['bloque_muestra'] = (
                f'{clave[0]}-{clave[1]}-{int(clave[2]):02d}-B{numero_bloque}'
            )
            seleccion.append(bloque)

    resultado = pd.concat(seleccion, ignore_index=True)
    resultado = resultado.drop_duplicates(subset='archivo')
    return resultado.sort_values(
        ['departamento', 'anio', 'mes', 'parte']
    ).reset_index(drop=True)


def cargar_muestra(seleccion):
    if seleccion.empty:
        return pd.DataFrame()

    filas_estimadas = int(seleccion['filas'].fillna(0).sum())
    if filas_estimadas > int(MAX_FILAS_MUESTRA):
        raise RuntimeError(
            f'La muestra seleccionada tendría {filas_estimadas:,} filas, '
            f'más que MAX_FILAS_MUESTRA={MAX_FILAS_MUESTRA:,}. '
            'Reduzca bloques, archivos por bloque o el alcance temporal.'
        )

    bloques = []
    for fila in seleccion.itertuples(index=False):
        columnas_archivo = list(fila.columnas)
        columnas_lectura = [
            columna for columna in COLUMNAS_ESPERADAS
            if columna in columnas_archivo
        ]
        bloque = pd.read_parquet(fila.archivo, columns=columnas_lectura)

        for columna in COLUMNAS_ESPERADAS:
            if columna not in bloque.columns:
                bloque[columna] = pd.NA

        bloque = bloque[COLUMNAS_ESPERADAS]
        bloque['_archivo'] = fila.archivo
        bloque['_bloque_muestra'] = fila.bloque_muestra
        bloque['_departamento_particion'] = fila.departamento
        bloque['_anio_particion'] = fila.anio
        bloque['_mes_particion'] = fila.mes
        bloques.append(bloque)

    return pd.concat(bloques, ignore_index=True)


def preparar_muestra(muestra):
    preparada = muestra.copy()

    fecha_original_valida = preparada['fechaobservacion'].notna()
    valor_original_valido = preparada['valorobservado'].notna()
    latitud_original_valida = preparada['latitud'].notna()
    longitud_original_valida = preparada['longitud'].notna()

    preparada['fechaobservacion_dt'] = pd.to_datetime(
        preparada['fechaobservacion'],
        errors='coerce',
    )
    preparada['valorobservado_num'] = pd.to_numeric(
        preparada['valorobservado'],
        errors='coerce',
    )
    preparada['latitud_num'] = pd.to_numeric(preparada['latitud'], errors='coerce')
    preparada['longitud_num'] = pd.to_numeric(preparada['longitud'], errors='coerce')

    preparada['_error_fecha'] = (
        fecha_original_valida & preparada['fechaobservacion_dt'].isna()
    )
    preparada['_error_valor'] = (
        valor_original_valido & preparada['valorobservado_num'].isna()
    )
    preparada['_error_latitud'] = (
        latitud_original_valida & preparada['latitud_num'].isna()
    )
    preparada['_error_longitud'] = (
        longitud_original_valida & preparada['longitud_num'].isna()
    )

    return preparada


## 5. Calidad de campos, fechas, duplicados y conflictos

La clave técnica provisional es:

codigoestacion + codigosensor + fechaobservacion

Se distinguen tres síntomas:

- Duplicado exacto: las 13 columnas coinciden.
- Duplicado de clave: la clave técnica aparece más de una vez.
- Conflicto: una clave repetida contiene valores observados diferentes.

Los Parquet exportados conservan **todas** las filas y claves detectadas en la muestra. El Markdown solo presenta hasta 50 ejemplos para mantenerse legible.

También se contrasta el año y mes internos de `fechaobservacion` con la carpeta de la partición. La auditoría reporta estos casos; no elimina ni corrige filas.


In [ ]:
CLAVE_OBSERVACION = [
    'codigoestacion',
    'codigosensor',
    'fechaobservacion_dt',
]


def auditar_nulos(muestra):
    total = len(muestra)
    return pd.DataFrame([
        {
            'columna': columna,
            'nulos': int(muestra[columna].isna().sum()),
            'porcentaje_nulos': round(
                muestra[columna].isna().mean() * 100,
                2,
            ) if total else 0.0,
        }
        for columna in COLUMNAS_ESPERADAS
    ])


def auditar_conversiones(muestra):
    return pd.DataFrame([
        {'campo': 'fechaobservacion', 'conversiones_fallidas': int(muestra['_error_fecha'].sum())},
        {'campo': 'valorobservado', 'conversiones_fallidas': int(muestra['_error_valor'].sum())},
        {'campo': 'latitud', 'conversiones_fallidas': int(muestra['_error_latitud'].sum())},
        {'campo': 'longitud', 'conversiones_fallidas': int(muestra['_error_longitud'].sum())},
    ])


def auditar_fechas_particion(muestra):
    validas = muestra['fechaobservacion_dt'].notna()
    anio_observado = muestra['fechaobservacion_dt'].dt.year
    mes_observado = muestra['fechaobservacion_dt'].dt.month
    anio_fuera = validas & anio_observado.ne(muestra['_anio_particion'])
    mes_fuera = validas & mes_observado.ne(muestra['_mes_particion'])
    fuera = anio_fuera | mes_fuera

    resumen = pd.DataFrame([
        {'metrica': 'fechas_validas_muestra', 'registros_muestra': int(validas.sum())},
        {'metrica': 'anio_distinto_particion', 'registros_muestra': int(anio_fuera.sum())},
        {'metrica': 'mes_distinto_particion', 'registros_muestra': int(mes_fuera.sum())},
        {'metrica': 'fecha_fuera_particion', 'registros_muestra': int(fuera.sum())},
    ])
    columnas = [
        'codigoestacion', 'codigosensor', 'fechaobservacion',
        'fechaobservacion_dt', '_departamento_particion',
        '_anio_particion', '_mes_particion', '_archivo',
    ]
    return resumen, muestra.loc[fuera, columnas].copy()


def auditar_coordenadas_y_particion(muestra):
    latitud_invalida = muestra['latitud_num'].notna() & ~muestra['latitud_num'].between(-90, 90)
    longitud_invalida = muestra['longitud_num'].notna() & ~muestra['longitud_num'].between(-180, 180)
    departamento_dato = (
        muestra['departamento'].fillna('<NULO>').astype(str).str.strip().str.upper().str.normalize('NFC')
    )
    departamento_particion = (
        muestra['_departamento_particion'].fillna('<NULO>').astype(str).str.strip().str.upper().str.normalize('NFC')
    )
    departamento_inconsistente = departamento_dato != departamento_particion

    return pd.DataFrame([
        {'metrica': 'latitudes_fuera_rango', 'registros_muestra': int(latitud_invalida.sum())},
        {'metrica': 'longitudes_fuera_rango', 'registros_muestra': int(longitud_invalida.sum())},
        {
            'metrica': 'departamento_distinto_particion',
            'registros_muestra': int(departamento_inconsistente.sum()),
        },
    ])


def auditar_unidades(muestra):
    return (
        muestra['unidadmedida']
        .fillna('<NULO>')
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis('unidadmedida')
        .reset_index(name='registros_muestra')
    )


def auditar_valores(muestra):
    valores = muestra['valorobservado_num'].dropna()
    if valores.empty:
        return pd.DataFrame(), pd.DataFrame()

    resumen = pd.DataFrame([{
        'registros_validos': len(valores),
        'minimo': valores.min(),
        'p01': valores.quantile(0.01),
        'p05': valores.quantile(0.05),
        'mediana': valores.median(),
        'media': valores.mean(),
        'p95': valores.quantile(0.95),
        'p99': valores.quantile(0.99),
        'maximo': valores.max(),
    }])

    sospechosos = pd.DataFrame()
    if RANGO_PLAUSIBLE is not None:
        minimo, maximo = RANGO_PLAUSIBLE
        mascara = (muestra['valorobservado_num'] < minimo) | (
            muestra['valorobservado_num'] > maximo
        )
        sospechosos = muestra.loc[
            mascara,
            COLUMNAS_ESPERADAS + ['valorobservado_num', '_archivo'],
        ].copy()

    return resumen, sospechosos


def auditar_duplicados(muestra):
    columnas_exactas = [
        columna for columna in COLUMNAS_ESPERADAS
        if columna in muestra.columns
    ]
    mascara_exactos = muestra.duplicated(
        subset=columnas_exactas,
        keep=False,
    )
    duplicados_exactos = (
        muestra.loc[mascara_exactos, columnas_exactas + ['_archivo']]
        .sort_values(columnas_exactas[:3])
        .reset_index(drop=True)
    )

    claves_validas = muestra.dropna(subset=CLAVE_OBSERVACION)
    grupos = (
        claves_validas
        .groupby(CLAVE_OBSERVACION, dropna=False)
        .agg(
            registros=('valorobservado_num', 'size'),
            valores_distintos=('valorobservado_num', lambda serie: serie.nunique(dropna=False)),
            valor_min=('valorobservado_num', 'min'),
            valor_max=('valorobservado_num', 'max'),
        )
        .reset_index()
    )

    claves_duplicadas = (
        grupos[grupos['registros'] > 1]
        .sort_values(['registros', *CLAVE_OBSERVACION], ascending=[False, True, True, True])
        .reset_index(drop=True)
    )
    conflictos = (
        claves_duplicadas[claves_duplicadas['valores_distintos'] > 1]
        .reset_index(drop=True)
    )

    resumen = pd.DataFrame([{
        'filas_duplicadas_exactas': int(mascara_exactos.sum()),
        'grupos_duplicados_exactos': int(
            muestra.loc[mascara_exactos, columnas_exactas]
            .drop_duplicates()
            .shape[0]
        ),
        'claves_duplicadas': len(claves_duplicadas),
        'claves_con_valores_conflictivos': len(conflictos),
    }])

    return resumen, duplicados_exactos, claves_duplicadas, conflictos


## 6. Estaciones, sensores, frecuencia y cobertura

Los intervalos se calculan dentro de cada bloque contiguo para no introducir saltos artificiales entre muestras. El reporte incluye un resumen de **todas** las cadencias encontradas y separa las que no estén en `CADENCIAS_ESPERADAS_SEGUNDOS`.

La tabla de fragmentos diarios describe únicamente lo que cayó dentro de los bloques muestreados. No representa cobertura diaria real.

El conteo completo opcional escanea `codigoestacion`, `codigosensor` y `fechaobservacion`. Cuando se activa, ofrece conteos, rangos y estaciones activas mensuales exactos para las particiones seleccionadas, pero puede tardar con variables voluminosas.


In [ ]:
import math


def resumir_estaciones_muestra(muestra):
    validos = muestra.dropna(
        subset=['codigoestacion', 'codigosensor', 'fechaobservacion_dt']
    ).copy()
    if validos.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    claves_sensor = ['codigoestacion', 'codigosensor']

    conteos = (
        validos
        .groupby(claves_sensor, dropna=False)
        .agg(
            registros_muestra=('fechaobservacion_dt', 'size'),
            fecha_min_muestra=('fechaobservacion_dt', 'min'),
            fecha_max_muestra=('fechaobservacion_dt', 'max'),
            dias_observados_muestra=('fechaobservacion_dt', lambda serie: serie.dt.date.nunique()),
            unidades_muestra=('unidadmedida', lambda serie: serie.nunique(dropna=False)),
            municipios_muestra=('municipio', lambda serie: serie.nunique(dropna=False)),
        )
        .reset_index()
    )
    conteos['alcance'] = 'muestra'

    ordenados = validos.sort_values(
        ['_bloque_muestra', 'codigoestacion', 'codigosensor', 'fechaobservacion_dt']
    ).copy()
    ordenados['delta_segundos'] = (
        ordenados
        .groupby(
            ['_bloque_muestra', 'codigoestacion', 'codigosensor'],
            dropna=False,
        )['fechaobservacion_dt']
        .diff()
        .dt.total_seconds()
    )

    positivos = ordenados[ordenados['delta_segundos'] > 0].copy()

    def moda_segundos(serie):
        modas = serie.mode()
        return modas.iloc[0] if not modas.empty else pd.NA

    if positivos.empty:
        frecuencia = pd.DataFrame(columns=[
            *claves_sensor, 'intervalos_muestra', 'intervalo_moda_segundos',
            'intervalo_mediano_segundos', 'intervalo_p10_segundos',
            'intervalo_p90_segundos', 'intervalo_min_segundos',
            'intervalo_max_segundos', 'alcance',
        ])
    else:
        frecuencia = (
            positivos
            .groupby(claves_sensor, dropna=False)
            .agg(
                intervalos_muestra=('delta_segundos', 'size'),
                intervalo_moda_segundos=('delta_segundos', moda_segundos),
                intervalo_mediano_segundos=('delta_segundos', 'median'),
                intervalo_p10_segundos=('delta_segundos', lambda serie: serie.quantile(0.10)),
                intervalo_p90_segundos=('delta_segundos', lambda serie: serie.quantile(0.90)),
                intervalo_min_segundos=('delta_segundos', 'min'),
                intervalo_max_segundos=('delta_segundos', 'max'),
            )
            .reset_index()
        )
        frecuencia['alcance'] = 'muestra_bloques_contiguos'

    por_dia = validos.copy()
    por_dia['fecha_dia'] = por_dia['fechaobservacion_dt'].dt.floor('D')
    por_dia = (
        por_dia
        .groupby(claves_sensor + ['fecha_dia'], dropna=False)
        .size()
        .reset_index(name='observaciones_fragmento_dia')
    )
    fragmentos_diarios = (
        por_dia
        .groupby(claves_sensor, dropna=False)
        .agg(
            dias_en_fragmentos=('fecha_dia', 'size'),
            observaciones_fragmento_min=('observaciones_fragmento_dia', 'min'),
            observaciones_fragmento_mediana=('observaciones_fragmento_dia', 'median'),
            observaciones_fragmento_max=('observaciones_fragmento_dia', 'max'),
        )
        .reset_index()
    )
    fragmentos_diarios['alcance'] = 'fragmentos_muestrales_no_cobertura'

    return conteos, frecuencia, fragmentos_diarios


def resumir_cadencias(frecuencia):
    columnas = [
        'codigosensor', 'intervalo_moda_segundos',
        'pares_estacion_sensor', 'intervalos_muestra',
    ]
    if frecuencia.empty:
        return pd.DataFrame(columns=columnas)

    return (
        frecuencia.dropna(subset=['intervalo_moda_segundos'])
        .groupby(['codigosensor', 'intervalo_moda_segundos'], as_index=False, dropna=False)
        .agg(
            pares_estacion_sensor=('codigoestacion', 'size'),
            intervalos_muestra=('intervalos_muestra', 'sum'),
        )
        .sort_values(['codigosensor', 'intervalo_moda_segundos'])
        .reset_index(drop=True)
    )


def detectar_cadencias_no_esperadas(frecuencia):
    columnas = list(frecuencia.columns)
    if frecuencia.empty or CADENCIAS_ESPERADAS_SEGUNDOS is None:
        return pd.DataFrame(columns=columnas)

    esperadas = {float(valor) for valor in CADENCIAS_ESPERADAS_SEGUNDOS}
    return (
        frecuencia[
            frecuencia['intervalo_moda_segundos'].notna()
            & ~frecuencia['intervalo_moda_segundos'].isin(esperadas)
        ]
        .sort_values(['intervalo_moda_segundos', 'codigoestacion', 'codigosensor'])
        .reset_index(drop=True)
    )


def valores_observados(serie):
    valores = sorted({
        normalizar_unicode(valor).strip()
        for valor in serie.dropna().astype(str)
    })
    return ' | '.join(valores) if valores else '<NULO>'


def auditar_geografia(muestra):
    base = muestra.dropna(subset=['codigoestacion']).copy()
    if base.empty:
        return pd.DataFrame()

    geografia = (
        base
        .groupby('codigoestacion', dropna=False)
        .agg(
            nombres_estacion=('nombreestacion', lambda serie: serie.nunique(dropna=False)),
            nombres_observados=('nombreestacion', valores_observados),
            departamentos=('departamento', lambda serie: serie.nunique(dropna=False)),
            departamentos_observados=('departamento', valores_observados),
            municipios=('municipio', lambda serie: serie.nunique(dropna=False)),
            municipios_observados=('municipio', valores_observados),
            latitud_min=('latitud_num', 'min'),
            latitud_max=('latitud_num', 'max'),
            longitud_min=('longitud_num', 'min'),
            longitud_max=('longitud_num', 'max'),
        )
        .reset_index()
    )

    latitud_media_rad = (
        (geografia['latitud_min'] + geografia['latitud_max']) / 2
    ).map(math.radians)
    delta_latitud_m = (geografia['latitud_max'] - geografia['latitud_min']) * 111_320
    delta_longitud_m = (
        (geografia['longitud_max'] - geografia['longitud_min'])
        * 111_320
        * latitud_media_rad.map(math.cos)
    )
    geografia['variacion_coordenadas_m'] = (
        delta_latitud_m.pow(2) + delta_longitud_m.pow(2)
    ).pow(0.5).round(2)
    geografia['alerta_geografica'] = (
        (geografia['departamentos'] > 1)
        | (geografia['municipios'] > 1)
        | (geografia['variacion_coordenadas_m'] > UMBRAL_VARIACION_COORDENADAS_METROS)
    )
    return geografia.sort_values(
        ['alerta_geografica', 'variacion_coordenadas_m'],
        ascending=[False, False],
    ).reset_index(drop=True)


def resumir_estaciones_mes_muestra(muestra):
    validos = muestra.dropna(
        subset=['codigoestacion', 'codigosensor', 'fechaobservacion_dt']
    ).copy()
    if validos.empty:
        return pd.DataFrame()

    return (
        validos
        .groupby(
            ['_departamento_particion', '_anio_particion', '_mes_particion',
             'codigoestacion', 'codigosensor'],
            as_index=False,
            dropna=False,
        )
        .agg(
            registros=('fechaobservacion_dt', 'size'),
            fecha_min=('fechaobservacion_dt', 'min'),
            fecha_max=('fechaobservacion_dt', 'max'),
        )
        .rename(columns={
            '_departamento_particion': 'departamento',
            '_anio_particion': 'anio',
            '_mes_particion': 'mes',
        })
        .assign(alcance='muestra')
    )


def _actualizar_acumulado(acumulado, clave, fila):
    if clave not in acumulado:
        acumulado[clave] = {
            'registros': 0,
            'fecha_min': fila.fecha_min,
            'fecha_max': fila.fecha_max,
        }

    actual = acumulado[clave]
    actual['registros'] += int(fila.registros)
    if pd.notna(fila.fecha_min):
        if pd.isna(actual['fecha_min']) or fila.fecha_min < actual['fecha_min']:
            actual['fecha_min'] = fila.fecha_min
    if pd.notna(fila.fecha_max):
        if pd.isna(actual['fecha_max']) or fila.fecha_max > actual['fecha_max']:
            actual['fecha_max'] = fila.fecha_max


def contar_estaciones_completo(archivos, raiz):
    acumulado_total = {}
    acumulado_mes = {}

    try:
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise ImportError('Se necesita pyarrow para el conteo completo.') from exc

    for numero, archivo in enumerate(archivos, start=1):
        try:
            columnas = set(pq.ParquetFile(archivo).schema_arrow.names)
            requeridas = {'codigoestacion', 'codigosensor', 'fechaobservacion'}
            if not requeridas.issubset(columnas):
                continue

            particion = particiones_desde_ruta(archivo, raiz)
            bloque = pd.read_parquet(
                archivo,
                columns=['codigoestacion', 'codigosensor', 'fechaobservacion'],
            )
            bloque['fechaobservacion'] = pd.to_datetime(
                bloque['fechaobservacion'],
                errors='coerce',
            )
            for columna in ['codigoestacion', 'codigosensor']:
                bloque[columna] = bloque[columna].astype('string').fillna('<NULO>')

            parcial = (
                bloque
                .groupby(['codigoestacion', 'codigosensor'], dropna=False)
                .agg(
                    registros=('fechaobservacion', 'size'),
                    fecha_min=('fechaobservacion', 'min'),
                    fecha_max=('fechaobservacion', 'max'),
                )
                .reset_index()
            )

            for fila in parcial.itertuples(index=False):
                clave_total = (fila.codigoestacion, fila.codigosensor)
                clave_mes = (
                    particion['departamento'], particion['anio'], particion['mes'],
                    fila.codigoestacion, fila.codigosensor,
                )
                _actualizar_acumulado(acumulado_total, clave_total, fila)
                _actualizar_acumulado(acumulado_mes, clave_mes, fila)

        except Exception as exc:
            print(f'ADVERTENCIA al contar {archivo}: {type(exc).__name__}: {exc}')

        if numero == 1 or numero % 250 == 0 or numero == len(archivos):
            print(f'Conteo completo: {numero:,}/{len(archivos):,} archivos.')

    total = pd.DataFrame([
        {
            'codigoestacion': clave[0],
            'codigosensor': clave[1],
            **valores,
            'alcance': 'completo',
        }
        for clave, valores in acumulado_total.items()
    ])
    detalle_mes = pd.DataFrame([
        {
            'departamento': clave[0],
            'anio': clave[1],
            'mes': clave[2],
            'codigoestacion': clave[3],
            'codigosensor': clave[4],
            **valores,
            'alcance': 'completo',
        }
        for clave, valores in acumulado_mes.items()
    ])

    if not total.empty:
        total = total.sort_values('registros', ascending=False).reset_index(drop=True)
    if not detalle_mes.empty:
        detalle_mes = detalle_mes.sort_values(
            ['departamento', 'anio', 'mes', 'codigoestacion', 'codigosensor']
        ).reset_index(drop=True)
    return total, detalle_mes


def resumir_actividad_mensual(conteo_estaciones_mes):
    columnas = [
        'departamento', 'anio', 'mes', 'estaciones_activas',
        'pares_estacion_sensor', 'registros', 'fecha_min', 'fecha_max', 'alcance',
    ]
    if conteo_estaciones_mes.empty:
        return pd.DataFrame(columns=columnas)

    return (
        conteo_estaciones_mes
        .groupby(['departamento', 'anio', 'mes'], as_index=False, dropna=False)
        .agg(
            estaciones_activas=('codigoestacion', 'nunique'),
            pares_estacion_sensor=('codigosensor', 'size'),
            registros=('registros', 'sum'),
            fecha_min=('fecha_min', 'min'),
            fecha_max=('fecha_max', 'max'),
            alcance=('alcance', 'first'),
        )
        .sort_values(['departamento', 'anio', 'mes'])
        .reset_index(drop=True)
    )


## 7. Hallazgos y severidad

Los hallazgos resumen síntomas observados. Una alerta basada en muestra no afirma que todo el dataset tenga el problema; indica qué debe confirmarse o resolverse antes del procesamiento diario.

Severidades:

- CRITICO: impide una transformación confiable.
- ADVERTENCIA: requiere una regla explícita o revisión.
- INFORMATIVO: describe el alcance o comportamiento.


In [ ]:
def construir_hallazgos(
    inventario,
    resumen_particiones,
    presencia_columnas,
    muestra,
    conversiones,
    coordenadas_particion,
    unidades,
    resumen_duplicados,
    frecuencia,
    geografia,
    valores_sospechosos,
    fechas_fuera_particion,
    cadencias_no_esperadas,
):
    hallazgos = []

    def agregar(severidad, categoria, metrica, valor, alcance, recomendacion):
        hallazgos.append({
            'severidad': severidad,
            'categoria': categoria,
            'metrica': metrica,
            'valor': str(valor),
            'alcance': alcance,
            'recomendacion': recomendacion,
        })

    agregar(
        'INFORMATIVO',
        'alcance',
        'archivos_auditados',
        len(inventario),
        'inventario_completo',
        'Conservar este valor como trazabilidad de la corrida.',
    )
    agregar(
        'INFORMATIVO',
        'alcance',
        'filas_inventariadas',
        int(inventario['filas'].fillna(0).sum()) if not inventario.empty else 0,
        'inventario_completo',
        'Conteo exacto obtenido desde metadatos Parquet.',
    )
    agregar(
        'INFORMATIVO',
        'alcance',
        'filas_muestra',
        len(muestra),
        'muestra_estratificada',
        'No interpretar conteos de la muestra como totales del dataset.',
    )

    errores_archivo = int(inventario['error_lectura'].notna().sum()) if not inventario.empty else 0
    if errores_archivo:
        agregar(
            'CRITICO',
            'archivos',
            'archivos_no_legibles',
            errores_archivo,
            'inventario_completo',
            'Reparar o volver a descargar los Parquet afectados.',
        )

    partes_con_huecos = 0
    if not resumen_particiones.empty:
        partes_con_huecos = int(
            resumen_particiones['partes_faltantes'].apply(bool).sum()
        )
    if partes_con_huecos:
        agregar(
            'CRITICO',
            'particiones',
            'particiones_con_huecos',
            partes_con_huecos,
            'inventario_completo',
            'No procesar hasta completar o justificar las partes faltantes.',
        )

    presentes = set(
        presencia_columnas.loc[
            presencia_columnas['archivos_presente'] > 0,
            'columna',
        ]
    ) if not presencia_columnas.empty else set()
    faltantes = sorted(set(COLUMNAS_ESPERADAS) - presentes)
    if faltantes:
        agregar(
            'CRITICO',
            'esquema',
            'columnas_esperadas_ausentes',
            ', '.join(faltantes),
            'inventario_completo',
            'Homologar el esquema antes de combinar variables o años.',
        )

    incompletas = []
    if not presencia_columnas.empty:
        incompletas = presencia_columnas.loc[
            presencia_columnas['esperada']
            & (presencia_columnas['archivos_presente'] < presencia_columnas['archivos_totales']),
            'columna',
        ].tolist()
    if incompletas:
        agregar(
            'CRITICO',
            'esquema',
            'columnas_ausentes_en_algunos_archivos',
            ', '.join(incompletas),
            'inventario_completo',
            'Homologar o volver a descargar los archivos con esquema incompleto.',
        )

    firmas = int(inventario['firma_esquema'].dropna().nunique()) if not inventario.empty else 0
    if firmas > 1:
        agregar(
            'ADVERTENCIA',
            'esquema',
            'variantes_de_esquema',
            firmas,
            'inventario_completo',
            'Revisar columnas y tipos por archivo antes de concatenar.',
        )

    fallos_conversion = int(conversiones['conversiones_fallidas'].sum()) if not conversiones.empty else 0
    if fallos_conversion:
        agregar(
            'CRITICO',
            'tipos',
            'conversiones_fallidas',
            fallos_conversion,
            'muestra_estratificada',
            'Definir reglas para fechas, valores o coordenadas no interpretables.',
        )

    problemas_coordenadas = 0
    if not coordenadas_particion.empty:
        problemas_coordenadas = int(coordenadas_particion['registros_muestra'].sum())
    if problemas_coordenadas:
        agregar(
            'CRITICO',
            'geografia',
            'coordenadas_o_particion_invalidas',
            problemas_coordenadas,
            'muestra_estratificada',
            'Corregir o excluir registros inválidos antes de asignar estaciones a municipios.',
        )

    if not fechas_fuera_particion.empty:
        agregar(
            'CRITICO',
            'fechas',
            'fechas_fuera_de_particion',
            len(fechas_fuera_particion),
            'muestra_estratificada',
            'Revisar la descarga: el año o mes interno no coincide con su carpeta.',
        )

    if not resumen_duplicados.empty:
        exactos = int(resumen_duplicados.iloc[0]['filas_duplicadas_exactas'])
        claves = int(resumen_duplicados.iloc[0]['claves_duplicadas'])
        conflictos = int(resumen_duplicados.iloc[0]['claves_con_valores_conflictivos'])

        if exactos:
            agregar(
                'ADVERTENCIA',
                'duplicados',
                'filas_duplicadas_exactas',
                exactos,
                'muestra_estratificada',
                'Definir deduplicación exacta antes de agregar por día.',
            )
        if claves:
            agregar(
                'ADVERTENCIA',
                'duplicados',
                'claves_repetidas',
                claves,
                'muestra_estratificada',
                'Revisar estación, sensor y timestamp antes de escoger un registro.',
            )
        if conflictos:
            agregar(
                'CRITICO',
                'duplicados',
                'claves_con_valores_distintos',
                conflictos,
                'muestra_estratificada',
                'No promediar conflictos automáticamente; investigar su origen.',
            )

    if len(unidades) > 1:
        agregar(
            'ADVERTENCIA',
            'unidades',
            'unidades_distintas',
            len(unidades),
            'muestra_estratificada',
            'Separar o convertir unidades antes de cualquier agregación.',
        )

    if not frecuencia.empty:
        cadencias = int(frecuencia['intervalo_moda_segundos'].dropna().nunique())
        if cadencias > 1:
            agregar(
                'ADVERTENCIA',
                'frecuencia',
                'cadencias_modales_distintas',
                cadencias,
                'muestra_bloques_contiguos',
                'Definir cobertura diaria por sensor; no ponderar por número bruto de filas.',
            )

    if not cadencias_no_esperadas.empty:
        agregar(
            'ADVERTENCIA',
            'frecuencia',
            'pares_con_cadencia_no_esperada',
            len(cadencias_no_esperadas),
            'muestra_bloques_contiguos',
            'Verificar semántica y configuración antes de procesar estos sensores.',
        )

    if not geografia.empty:
        estaciones_inconsistentes = int(geografia['alerta_geografica'].sum())
        if estaciones_inconsistentes:
            agregar(
                'ADVERTENCIA',
                'geografia',
                'estaciones_con_geografia_variable',
                estaciones_inconsistentes,
                'muestra_estratificada',
                'Revisar traslados, etiquetas y coordenadas antes de asignar municipios.',
            )

    if not valores_sospechosos.empty:
        agregar(
            'ADVERTENCIA',
            'valores',
            'valores_fuera_rango_configurado',
            len(valores_sospechosos),
            'muestra_estratificada',
            'Validar unidad y sensor antes de excluir o marcar estos valores.',
        )

    if len(hallazgos) == 3:
        agregar(
            'INFORMATIVO',
            'resultado',
            'sin_alertas_en_muestra',
            True,
            'muestra_estratificada',
            'La ausencia de alertas en una muestra no demuestra ausencia de problemas.',
        )

    orden = {'CRITICO': 0, 'ADVERTENCIA': 1, 'INFORMATIVO': 2}
    resultado = pd.DataFrame(hallazgos)
    resultado['_orden'] = resultado['severidad'].map(orden)
    return resultado.sort_values(
        ['_orden', 'categoria', 'metrica']
    ).drop(columns='_orden').reset_index(drop=True)


def etiqueta_automatica_auditoria():
    departamentos = '-'.join(
        slugificar(valor) for valor in sorted(AUDITORIA_DEPARTAMENTOS)
    )
    if AUDITORIA_ANIOS is None:
        periodo = 'todos_los_anios'
    else:
        anios = sorted({int(valor) for valor in AUDITORIA_ANIOS})
        periodo = str(anios[0]) if len(anios) == 1 else f'{anios[0]}_{anios[-1]}'

    modo = (
        'conteo_completo'
        if EJECUTAR_CONTEO_ESTACIONES_COMPLETO
        else 'muestra'
    )
    return slugificar(
        f'{VARIABLE_NOMBRE}_{DATASET_ID}_{departamentos}_{periodo}_{modo}'
    )


def tabla_como_markdown(tabla, max_filas=50):
    if not isinstance(tabla, pd.DataFrame) or tabla.empty:
        return '_Sin registros._'

    recorte = tabla.head(max_filas)
    try:
        contenido = recorte.to_markdown(index=False)
    except ImportError:
        contenido = f'```text\n{recorte.to_string(index=False)}\n```'

    if len(tabla) > max_filas:
        contenido += f'\n\n_Mostradas {max_filas} de {len(tabla)} filas._'
    return contenido



def construir_reporte_markdown(tablas, etiqueta):
    from datetime import datetime

    inventario = tablas.get('inventario_archivos', pd.DataFrame())
    muestra = tablas.get('muestra_resumen', pd.DataFrame())
    total_archivos = len(inventario)
    total_filas = int(inventario['filas'].fillna(0).sum()) if not inventario.empty else 0
    total_mb = (
        inventario['tamano_bytes'].fillna(0).sum() / (1024 ** 2)
        if not inventario.empty
        else 0
    )
    filas_muestra = int(muestra.iloc[0]['filas_muestra']) if not muestra.empty else 0

    secciones = [
        f'# Auditoría climática: {VARIABLE_NOMBRE}',
        '',
        '## Identificación',
        '',
        f'- Fuente: `{DATASET_ID}`.',
        f"- Departamentos: {', '.join(AUDITORIA_DEPARTAMENTOS)}.",
        f'- Años: {AUDITORIA_ANIOS if AUDITORIA_ANIOS is not None else "todos"}.',
        f'- Meses: {AUDITORIA_MESES if AUDITORIA_MESES is not None else "todos los disponibles"}.',
        f'- Modo: {MODO_EJECUCION}.',
        f'- Etiqueta: `{etiqueta}`.',
        f'- Generado: {datetime.now():%Y-%m-%d %H:%M:%S}.',
        '',
        '## Alcance',
        '',
        f'- Archivos Parquet: {total_archivos:,}.',
        f'- Filas exactas por metadatos: {total_filas:,}.',
        f'- Tamaño: {total_mb:,.2f} MB.',
        f'- Filas realmente cargadas en la muestra: {filas_muestra:,}.',
        '',
        '## Particiones y esquema',
        '',
        '### Particiones',
        '',
        tabla_como_markdown(tablas.get('resumen_particiones')),
        '',
        '### Presencia de columnas',
        '',
        tabla_como_markdown(tablas.get('presencia_columnas')),
        '',
        '### Tipos Parquet',
        '',
        tabla_como_markdown(tablas.get('tipos_columnas')),
        '',
        '## Calidad muestral',
        '',
        '### Nulos',
        '',
        tabla_como_markdown(tablas.get('nulos')),
        '',
        '### Conversiones',
        '',
        tabla_como_markdown(tablas.get('conversiones')),
        '',
        '### Fecha contra partición',
        '',
        tabla_como_markdown(tablas.get('resumen_fechas_particion')),
        '',
        '### Ejemplos de fechas fuera de partición',
        '',
        tabla_como_markdown(tablas.get('fechas_fuera_particion')),
        '',
        '### Unidades',
        '',
        tabla_como_markdown(tablas.get('unidades')),
        '',
        '### Valores',
        '',
        tabla_como_markdown(tablas.get('resumen_valores')),
        '',
        '## Duplicados y conflictos',
        '',
        tabla_como_markdown(tablas.get('resumen_duplicados')),
        '',
        '### Filas duplicadas exactas (vista)',
        '',
        tabla_como_markdown(tablas.get('duplicados_exactos')),
        '',
        '### Claves repetidas (vista)',
        '',
        tabla_como_markdown(tablas.get('claves_duplicadas')),
        '',
        '### Conflictos (vista)',
        '',
        tabla_como_markdown(tablas.get('conflictos')),
        '',
        'Las tablas Parquet conservan todos los registros detectados; el reporte ',
        'Markdown muestra como máximo 50 filas por tabla.',
        '',
        '## Estaciones y frecuencia',
        '',
        '### Actividad mensual',
        '',
        tabla_como_markdown(tablas.get('actividad_mensual')),
        '',
        '### Resumen completo de cadencias muestrales',
        '',
        tabla_como_markdown(tablas.get('resumen_cadencias')),
        '',
        '### Sensores con cadencia no esperada',
        '',
        tabla_como_markdown(tablas.get('cadencias_no_esperadas')),
        '',
        '### Frecuencia por estación y sensor (vista)',
        '',
        tabla_como_markdown(tablas.get('frecuencia_estaciones'), max_filas=30),
        '',
        '### Conteo por estación y sensor (vista)',
        '',
        tabla_como_markdown(tablas.get('conteo_estaciones'), max_filas=30),
        '',
        '### Fragmentos diarios muestrales',
        '',
        'Esta tabla no representa cobertura diaria real.',
        '',
        tabla_como_markdown(tablas.get('fragmentos_diarios_muestra'), max_filas=30),
        '',
        '## Geografía muestral',
        '',
        f'Se alerta una variación de coordenadas superior a {UMBRAL_VARIACION_COORDENADAS_METROS} m, ',
        'o cambios de departamento o municipio.',
        '',
        tabla_como_markdown(tablas.get('geografia_alertas')),
        '',
        '## Hallazgos',
        '',
        tabla_como_markdown(tablas.get('hallazgos')),
        '',
        '## Nota metodológica',
        '',
        'El inventario de filas es exacto por metadatos. Calidad, duplicados, ',
        'frecuencia y geografía se diagnostican sobre una muestra estratificada. ',
        'El conteo completo opcional solo vuelve exactos los registros, rangos ',
        'temporales y actividad mensual por estación/sensor.',
    ]
    return '\n'.join(secciones) + '\n'
def guardar_tablas(tablas):
    etiqueta = (
        slugificar(ETIQUETA_SALIDA)
        if ETIQUETA_SALIDA
        else etiqueta_automatica_auditoria()
    )
    salida = (
        PROCESSED_ROOT
        / 'auditorias_climaticas'
        / f'variable={slugificar(VARIABLE_NOMBRE)}'
        / f'fuente={str(DATASET_ID).lower()}'
        / f'ejecucion={etiqueta}'
    )
    salida.mkdir(parents=True, exist_ok=True)

    for nombre, tabla in tablas.items():
        if isinstance(tabla, pd.DataFrame) and not tabla.empty:
            tabla.to_parquet(salida / f'{nombre}.parquet', index=False)

    nombre_reporte = f'AuditoriaClimatica_{etiqueta}.md'
    ruta_reporte = salida / nombre_reporte
    ruta_reporte.write_text(
        construir_reporte_markdown(tablas, etiqueta),
        encoding='utf-8',
    )

    print(f'Resultados guardados en: {salida}')
    print(f'Reporte Markdown: {ruta_reporte}')
    return salida, ruta_reporte


## 8. Ejecución

La ejecución muestra primero el inventario exacto y después los resultados de la muestra. Los Parquet exportados conservan las tablas completas; en pantalla y Markdown se presentan resúmenes y vistas controladas.

Para 2021 y 2023 use auditoría muestral (`EJECUTAR_CONTEO_ESTACIONES_COMPLETO=False`). El conteo completo de 2025 ya cumplió su propósito y no necesita repetirse.


In [ ]:
tablas_auditoria = {}
hallazgos = pd.DataFrame()

departamentos, anios, meses = validar_configuracion()
raiz = raiz_dataset()

print(f'Ruta auditada: {raiz}')
print(f'Auditoría activada: {EJECUTAR_AUDITORIA}')

if not EJECUTAR_AUDITORIA:
    print(
        'Auditoría desactivada. Para una muestra histórica, cambie únicamente '
        'EJECUTAR_AUDITORIA a True y conserve el conteo completo en False.'
    )
else:
    archivos = descubrir_archivos(
        raiz=raiz,
        departamentos=departamentos,
        anios=anios,
        meses=meses,
    )
    if not archivos:
        raise FileNotFoundError(
            'No se encontraron Parquet para los filtros configurados. '
            f'Ruta base: {raiz}'
        )

    display(Markdown('## Inventario exacto'))
    inventario_archivos = construir_inventario(archivos, raiz)
    resumen_particiones = resumir_particiones(inventario_archivos)
    presencia_columnas, tipos_columnas = resumir_esquemas(inventario_archivos)

    print(f'Archivos: {len(inventario_archivos):,}')
    print(f"Filas: {int(inventario_archivos['filas'].fillna(0).sum()):,}")
    print(
        'Tamaño: '
        f"{inventario_archivos['tamano_bytes'].fillna(0).sum() / (1024 ** 2):,.2f} MB"
    )
    display(resumen_particiones.head(60))
    display(presencia_columnas)
    display(tipos_columnas)

    display(Markdown('## Muestra estratificada'))
    seleccion_muestra = seleccionar_archivos_muestra(inventario_archivos)
    print(f'Archivos seleccionados: {len(seleccion_muestra):,}')
    print(f"Filas estimadas: {int(seleccion_muestra['filas'].fillna(0).sum()):,}")
    display(
        seleccion_muestra[
            ['departamento', 'anio', 'mes', 'parte', 'filas', 'bloque_muestra']
        ].head(60)
    )

    muestra_cruda = cargar_muestra(seleccion_muestra)
    muestra = preparar_muestra(muestra_cruda)
    muestra_resumen = pd.DataFrame([{
        'archivos_muestra': len(seleccion_muestra),
        'filas_muestra': len(muestra),
        'bloques_muestra': seleccion_muestra['bloque_muestra'].nunique(),
        'alcance': 'muestra_estratificada',
    }])
    print(f'Filas cargadas en muestra: {len(muestra):,}')

    nulos = auditar_nulos(muestra)
    conversiones = auditar_conversiones(muestra)
    resumen_fechas_particion, fechas_fuera_particion = auditar_fechas_particion(muestra)
    coordenadas_particion = auditar_coordenadas_y_particion(muestra)
    unidades = auditar_unidades(muestra)
    resumen_valores, valores_sospechosos = auditar_valores(muestra)
    (
        resumen_duplicados,
        duplicados_exactos,
        claves_duplicadas,
        conflictos,
    ) = auditar_duplicados(muestra)

    (
        conteo_estaciones_muestra,
        frecuencia_estaciones,
        fragmentos_diarios_muestra,
    ) = resumir_estaciones_muestra(muestra)
    resumen_cadencias = resumir_cadencias(frecuencia_estaciones)
    cadencias_no_esperadas = detectar_cadencias_no_esperadas(frecuencia_estaciones)
    geografia_estaciones = auditar_geografia(muestra)
    geografia_alertas = geografia_estaciones[
        geografia_estaciones['alerta_geografica']
    ].reset_index(drop=True)

    conteo_estaciones = conteo_estaciones_muestra
    conteo_estaciones_mes = resumir_estaciones_mes_muestra(muestra)
    if EJECUTAR_CONTEO_ESTACIONES_COMPLETO:
        conteo_estaciones, conteo_estaciones_mes = contar_estaciones_completo(
            archivos,
            raiz,
        )
    actividad_mensual = resumir_actividad_mensual(conteo_estaciones_mes)

    hallazgos = construir_hallazgos(
        inventario=inventario_archivos,
        resumen_particiones=resumen_particiones,
        presencia_columnas=presencia_columnas,
        muestra=muestra,
        conversiones=conversiones,
        coordenadas_particion=coordenadas_particion,
        unidades=unidades,
        resumen_duplicados=resumen_duplicados,
        frecuencia=frecuencia_estaciones,
        geografia=geografia_estaciones,
        valores_sospechosos=valores_sospechosos,
        fechas_fuera_particion=fechas_fuera_particion,
        cadencias_no_esperadas=cadencias_no_esperadas,
    )

    display(Markdown('## Calidad y valores'))
    display(nulos)
    display(conversiones)
    display(resumen_fechas_particion)
    display(coordenadas_particion)
    display(unidades)
    display(resumen_valores)
    display(resumen_duplicados)

    display(Markdown('## Estaciones y frecuencia'))
    display(actividad_mensual)
    display(resumen_cadencias)
    if not cadencias_no_esperadas.empty:
        display(Markdown('### Cadencias no esperadas'))
        display(cadencias_no_esperadas.head(30))
    display(conteo_estaciones.head(30))
    display(frecuencia_estaciones.head(30))

    display(Markdown('### Fragmentos diarios muestrales, no cobertura real'))
    display(fragmentos_diarios_muestra.head(30))

    display(Markdown('## Geografía'))
    print(
        f'Estaciones sobre {UMBRAL_VARIACION_COORDENADAS_METROS} m o con etiquetas variables: '
        f'{len(geografia_alertas):,}'
    )
    display(geografia_alertas.head(30))

    display(Markdown('## Hallazgos'))
    display(hallazgos)

    tablas_auditoria = {
        'inventario_archivos': inventario_archivos.drop(
            columns=['columnas', 'esquema'],
            errors='ignore',
        ),
        'resumen_particiones': resumen_particiones,
        'presencia_columnas': presencia_columnas,
        'tipos_columnas': tipos_columnas,
        'seleccion_muestra': seleccion_muestra.drop(
            columns=['columnas', 'esquema'],
            errors='ignore',
        ),
        'muestra_resumen': muestra_resumen,
        'nulos': nulos,
        'conversiones': conversiones,
        'resumen_fechas_particion': resumen_fechas_particion,
        'fechas_fuera_particion': fechas_fuera_particion,
        'coordenadas_particion': coordenadas_particion,
        'unidades': unidades,
        'resumen_valores': resumen_valores,
        'resumen_duplicados': resumen_duplicados,
        'duplicados_exactos': duplicados_exactos,
        'claves_duplicadas': claves_duplicadas,
        'conflictos': conflictos,
        'conteo_estaciones': conteo_estaciones,
        'conteo_estaciones_mes': conteo_estaciones_mes,
        'actividad_mensual': actividad_mensual,
        'frecuencia_estaciones': frecuencia_estaciones,
        'resumen_cadencias': resumen_cadencias,
        'cadencias_no_esperadas': cadencias_no_esperadas,
        'fragmentos_diarios_muestra': fragmentos_diarios_muestra,
        'geografia_estaciones': geografia_estaciones,
        'geografia_alertas': geografia_alertas,
        'valores_sospechosos': valores_sospechosos,
        'hallazgos': hallazgos,
    }

    if GUARDAR_RESULTADOS:
        guardar_tablas(tablas_auditoria)


## 9. Interpretación y siguiente paso

Este notebook responde qué síntomas presenta la fuente y con qué alcance fueron observados. No decide automáticamente qué fila conservar ni cómo resumir una variable.

Antes de ejecutar 03_Climate_TemperaturaMinima_DailyProcessor deben quedar explícitas, por variable:

- La semántica de valorobservado y unidadmedida.
- La clave definitiva de deduplicación.
- El tratamiento de conflictos.
- Los rangos físicamente plausibles.
- La frecuencia o frecuencias esperadas por sensor.
- La cobertura mínima para aceptar un día.
- Las estadísticas diarias apropiadas.
- Las banderas de calidad que acompañarán cada agregado.

Los huecos del calendario y una posible imputación se evalúan después de crear la capa diaria. No se imputan observaciones subdiarias por defecto.


## 10. Inventario operativo independiente

Esta sección solo depende de la configuración. Queda protegida por `EJECUTAR_INVENTARIO_OPERATIVO=False`, por lo que **Run all no recorrerá `clima_crudo`** mientras la bandera siga desactivada.

El inventario revisa la estructura de carpetas sin abrir los Parquet. Por eso el conteo de filas es un máximo estimado (`archivos x 1.000`) y la presencia de una carpeta mensual no demuestra que sus observaciones internas estén completas.

Antes de activarlo, ajuste los años esperados y, cuando se conozcan, las rutas de EVA y DIVIPOLA.


In [ ]:
import unicodedata
from itertools import product

# Ajuste únicamente este bloque para la corrida de inventario.
INVENTARIO_DEPARTAMENTOS_ESPERADOS = ['BOYACÁ', 'CUNDINAMARCA']
INVENTARIO_ANIOS_ESPERADOS = list(range(2021, 2026))
INVENTARIO_MESES_ESPERADOS = list(range(1, 13))
INVENTARIO_FILAS_MAXIMAS_POR_ARCHIVO = 1_000

# Reemplace None por la ruta completa cuando esté disponible.
INVENTARIO_ARCHIVOS_CLAVE = {
    'eva_upra_2019_2025': None,
    'divipola_curada': None,
}


def normalizar_etiqueta_inventario(valor):
    return unicodedata.normalize('NFC', str(valor).strip())


def etiquetas_particion_inventario(archivo, raiz):
    etiquetas = {}
    for parte in archivo.relative_to(raiz).parts:
        if '=' not in parte:
            continue
        clave, valor = parte.split('=', 1)
        etiquetas[clave] = (
            normalizar_etiqueta_inventario(valor)
            if clave == 'departamento'
            else valor
        )
    return etiquetas


def construir_inventario_operativo():
    raiz = PROCESSED_ROOT / 'clima_crudo'
    departamentos = sorted({
        normalizar_etiqueta_inventario(valor.upper())
        for valor in INVENTARIO_DEPARTAMENTOS_ESPERADOS
    })
    anios = sorted({int(valor) for valor in INVENTARIO_ANIOS_ESPERADOS})
    meses = sorted({int(valor) for valor in INVENTARIO_MESES_ESPERADOS})

    if not raiz.exists():
        print(f'No existe la carpeta climática esperada: {raiz}')
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    registros = []
    patron = 'variable=*/fuente=*/departamento=*/anio=*/mes=*/part-*.parquet'
    for archivo in raiz.glob(patron):
        etiquetas = etiquetas_particion_inventario(archivo, raiz)
        try:
            anio = int(etiquetas.get('anio'))
            mes = int(etiquetas.get('mes'))
        except (TypeError, ValueError):
            continue

        registros.append({
            'variable': etiquetas.get('variable'),
            'fuente': etiquetas.get('fuente'),
            'departamento': etiquetas.get('departamento'),
            'anio': anio,
            'mes': mes,
            'archivo': str(archivo),
        })

    encontrados = pd.DataFrame(registros)
    if encontrados.empty:
        print(f'No se encontraron archivos part-*.parquet en {raiz}.')
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    lotes_particion = (
        encontrados
        .groupby(
            ['variable', 'fuente', 'departamento', 'anio', 'mes'],
            as_index=False,
            dropna=False,
        )
        .agg(lotes_archivos=('archivo', 'count'))
    )

    fuentes = (
        encontrados[['variable', 'fuente']]
        .drop_duplicates()
        .sort_values(['variable', 'fuente'])
    )
    esperados = []
    for fuente in fuentes.itertuples(index=False):
        for departamento, anio, mes in product(departamentos, anios, meses):
            esperados.append({
                'variable': fuente.variable,
                'fuente': fuente.fuente,
                'departamento': departamento,
                'anio': anio,
                'mes': mes,
            })

    cobertura = pd.DataFrame(esperados).merge(
        lotes_particion,
        on=['variable', 'fuente', 'departamento', 'anio', 'mes'],
        how='left',
    )
    cobertura['lotes_archivos'] = (
        cobertura['lotes_archivos'].fillna(0).astype(int)
    )
    cobertura['particion_presente'] = cobertura['lotes_archivos'] > 0

    resumen = (
        cobertura
        .groupby(['variable', 'fuente'], as_index=False)
        .agg(
            particiones_presentes=('particion_presente', 'sum'),
            particiones_esperadas=('particion_presente', 'size'),
            lotes_archivos=('lotes_archivos', 'sum'),
        )
    )
    resumen['particiones_faltantes'] = (
        resumen['particiones_esperadas'] - resumen['particiones_presentes']
    )
    resumen['cobertura_estructural_pct'] = (
        100 * resumen['particiones_presentes'] / resumen['particiones_esperadas']
    ).round(2)
    resumen['registros_estimados_max'] = (
        resumen['lotes_archivos'] * int(INVENTARIO_FILAS_MAXIMAS_POR_ARCHIVO)
    )

    detalle = []
    claves_detalle = ['variable', 'fuente', 'departamento', 'anio']
    for clave, grupo in cobertura.groupby(claves_detalle, sort=True):
        presentes = sorted(
            grupo.loc[grupo['particion_presente'], 'mes'].astype(int).tolist()
        )
        faltantes = sorted(set(meses) - set(presentes))
        detalle.append({
            **dict(zip(claves_detalle, clave)),
            'meses_presentes': ','.join(f'{mes:02d}' for mes in presentes) or '-',
            'meses_faltantes': ','.join(f'{mes:02d}' for mes in faltantes) or '-',
            'lotes_archivos': int(grupo['lotes_archivos'].sum()),
            'registros_estimados_max': int(
                grupo['lotes_archivos'].sum()
                * INVENTARIO_FILAS_MAXIMAS_POR_ARCHIVO
            ),
            'estado_estructura': 'COMPLETA' if not faltantes else 'INCOMPLETA',
        })

    archivos_clave = []
    for nombre, ruta_configurada in INVENTARIO_ARCHIVOS_CLAVE.items():
        if ruta_configurada is None:
            archivos_clave.append({
                'recurso': nombre,
                'ruta': None,
                'estado': 'RUTA_NO_CONFIGURADA',
                'tipo': None,
                'tamano_mb': None,
            })
            continue

        ruta = Path(ruta_configurada).expanduser()
        existe = ruta.exists()
        archivos_clave.append({
            'recurso': nombre,
            'ruta': str(ruta),
            'estado': 'DISPONIBLE' if existe else 'NO_ENCONTRADO',
            'tipo': (
                'archivo' if ruta.is_file()
                else 'carpeta' if ruta.is_dir()
                else None
            ),
            'tamano_mb': (
                round(ruta.stat().st_size / (1024 ** 2), 2)
                if ruta.is_file()
                else None
            ),
        })

    return (
        resumen.sort_values(['variable', 'fuente']).reset_index(drop=True),
        pd.DataFrame(detalle),
        pd.DataFrame(archivos_clave),
    )


resumen_inventario_clima = pd.DataFrame()
detalle_inventario_clima = pd.DataFrame()
inventario_archivos_clave = pd.DataFrame()

if not EJECUTAR_INVENTARIO_OPERATIVO:
    print(
        'Inventario operativo desactivado. Cambie únicamente '
        'EJECUTAR_INVENTARIO_OPERATIVO a True para ejecutarlo.'
    )
else:
    (
        resumen_inventario_clima,
        detalle_inventario_clima,
        inventario_archivos_clave,
    ) = construir_inventario_operativo()

    print('[RESUMEN_INVENTARIO_CLIMA]')
    display(resumen_inventario_clima)

    print('\n[DETALLE_DEPARTAMENTO_ANIO]')
    display(detalle_inventario_clima)

    print('\n[ARCHIVOS_CLAVE]')
    display(inventario_archivos_clave)

    print(
        '\nNota: COMPLETA solo significa que existen las 12 carpetas mensuales con '
        'al menos un Parquet; no valida filas, fechas ni cobertura interna.'
    )
